In [1]:
from matplotlib import pyplot as plt
import numpy as np
import collections
from collections import namedtuple
import time
import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

import pandas as pd

from tqdm.notebook import tqdm

In [2]:
print(torch.cuda.get_device_name())
print('torch:', torch.__version__)

NVIDIA GeForce RTX 4080 SUPER
torch: 2.9.1+cu128


# Small Seq Model

In [29]:
df1 = pd.DataFrame()

In [30]:
Config = namedtuple('Config', 'device, samples_count, batch_size, epochs_count, i_dim, hidden')
configs = [
    Config(device='cpu',   samples_count=10_000,     batch_size=100, epochs_count=10, i_dim=38, hidden=32),
    Config(device='cpu',   samples_count=1_000_000,  batch_size=10_000, epochs_count=10, i_dim=38, hidden=32),
    Config(device='cpu',   samples_count=10_000_000, batch_size=100_000, epochs_count=10, i_dim=38, hidden=32),
    Config(device='cpu',   samples_count=50_000_000, batch_size=1_000_000, epochs_count=10, i_dim=38, hidden=32), # slow!
    Config(device='cpu',   samples_count=10_000,     batch_size=100, epochs_count=10, i_dim=38, hidden=100_000),
    Config(device='cuda',  samples_count=10_000,     batch_size=100, epochs_count=10, i_dim=38, hidden=32),
    Config(device='cuda',  samples_count=1_000_000,  batch_size=10_000, epochs_count=10, i_dim=38, hidden=32),
    Config(device='cuda',  samples_count=10_000_000, batch_size=100_000, epochs_count=10, i_dim=38, hidden=32),
    Config(device='cuda',  samples_count=50_000_000, batch_size=1_000_000, epochs_count=10, i_dim=38, hidden=32),
    Config(device='cuda',  samples_count=10_000,     batch_size=100, epochs_count=10, i_dim=38, hidden=100_000),
]

In [31]:
for device, samples_count, batch_size, epochs_count, i_dim, hidden in tqdm(configs, desc='Config'):
    net = nn.Sequential(
        nn.Linear(i_dim, hidden),
        nn.Tanh(), 
        nn.Linear(hidden, 1)).to(device)
    params_count = sum([p.numel() for p in net.parameters()])
    
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
    X_b = torch.randn((samples_count, i_dim), device=device)
    y_true = torch.randn((samples_count, 1), device=device)
    loss_fn = F.mse_loss
    
    t_start = time.time()
    t_forwards = []
    t_backwards = []
    
    for epoch in tqdm(range(epochs_count), leave=False, desc='Epoch'):
        t_forward, t_backward = 0, 0
        
        for i in tqdm(range(0, samples_count, batch_size), leave=False, desc='Batch'):
            x = X_b[i:i+batch_size]
            y = y_true[i:i+batch_size]
            
            optimizer.zero_grad(set_to_none=True)
            
            t0 = time.time()
            
            pred = net(x)
            loss = loss_fn(pred, y)
            
            torch.cuda.synchronize()
            t1 = time.time()
            
            loss.backward()
            
            torch.cuda.synchronize()
            t2 = time.time()
            
            t_forward+=(t1-t0)
            t_backward+=(t2-t1)
            t_forwards.append(t_forward)
            t_backwards.append(t_backward)
    
    df_new_row = pd.DataFrame({
        'device': [device], 
        'samples_count': [samples_count], 
        'dataset_size': [samples_count * i_dim], 
        'batch_size': [batch_size], 
        'params_count': [params_count], 
        'epochs_count': [epochs_count],
        'duration': [time.time()-t_start], 
        'avg. forward': [np.array(t_forwards).mean()], 
        'avg. backward': [np.array(t_backwards).mean()]
    })
    df1 = pd.concat([df1, df_new_row], ignore_index=True)
    
df1

Config:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

Batch:   0%|          | 0/100 [00:00<?, ?it/s]

Batch:   0%|          | 0/100 [00:00<?, ?it/s]

Batch:   0%|          | 0/100 [00:00<?, ?it/s]

...

Batch:   0%|          | 0/100 [00:00<?, ?it/s]

Batch:   0%|          | 0/100 [00:00<?, ?it/s]

Batch:   0%|          | 0/100 [00:00<?, ?it/s]

Batch:   0%|          | 0/100 [00:00<?, ?it/s]

,device,samples_count,dataset_size,batch_size,params_count,epochs_count,duration,avg. forward,avg. backward
0,cpu,10000,380000,100,1281,10,0.839711,0.015223,0.017000
1,cpu,1000000,38000000,10000,1281,10,1.897092,0.026496,0.059301
2,cpu,10000000,380000000,100000,1281,10,6.271907,0.083632,0.218530
3,cpu,50000000,1900000000,1000000,1281,10,64.854122,1.196004,2.071281
4,cpu,10000,380000,100,4000001,10,66.573575,1.115222,2.195962
5,cuda,10000,380000,100,1281,10,2.486310,0.037656,0.065221
6,cuda,1000000,38000000,10000,1281,10,2.721744,0.042436,0.070916
7,cuda,10000000,380000000,100000,1281,10,2.750013,0.043710,0.070638
8,cuda,50000000,1900000000,1000000,1281,10,2.079400,0.043105,0.050989
9,cuda,10000,380000,100,4000001,10,2.553622,0.040564,0.064786


1) CPU быстрее, если соблюдены ДВА условия одновременно:
   - небольшой batch_size (до 100к)
   - в модели мало параметров (до неск. десятков тысяч)
2) если же или batch_size большой, или в модели много параметров (от 100к), (или и то и другое сразу), то GPU быстрее


# Larger Autoenc Model

In [6]:
df2 = pd.DataFrame()

In [7]:
class TestModel(nn.Module):
    def __init__(self, inp_dims_count, hiddens_count):
        super().__init__()
        self.filters = nn.Linear(inp_dims_count, hiddens_count)
        self.decoder = nn.Linear(hiddens_count, inp_dims_count)

        bound = 1 / np.sqrt(inp_dims_count)
        self.filters.weight.data.uniform_(-bound, bound)
        bound = 1 / np.sqrt(hiddens_count)
        self.decoder.weight.data.uniform_(-bound, bound)

    def forward(self, inp):
        out = F.sigmoid(self.filters(inp))
        out = self.decoder(out)
        return out

In [8]:
Config = namedtuple('Config', 'device, samples_count, i_dim, hidden')
configs = [
    Config('cpu',  100, 144, 200),
    Config('cpu',  10_000, 144, 200),
    Config('cpu',  100_000, 144, 200), # slow
    # Config('cpu',  1_000_000, 144, 200), # too slow!!
    Config('cuda',  100, 144, 200),
    Config('cuda',  10_000, 144, 200),
    Config('cuda',  100_000, 144, 200),
    Config('cuda',  1_000_000, 144, 200),
]

In [9]:
for device, samples_count, i_dim, hidden in tqdm(configs):
    net = TestModel(i_dim, hidden).to(device=device)
    params_count = sum([p.numel() for p in net.parameters()])
    
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
    X_b = torch.randn((samples_count, i_dim), device=device)
    y_true = torch.randn((samples_count, i_dim), device=device)
    loss_fn = F.mse_loss
    
    t_start = time.time()
    t_forwards = []
    t_backwards = []
    
    for epoch in tqdm(range(20), leave=False):
        t_forward=0
        t_backward=0
        
        for _ in tqdm(range(100), leave=False):
            optimizer.zero_grad(set_to_none=True)
            
            t0 = time.time()
            
            pred = net(X_b)
            loss = loss_fn(pred, y_true)
            
            torch.cuda.synchronize()
            t1 = time.time()
            
            loss.backward()
            
            torch.cuda.synchronize()
            t2 = time.time()
            
            t_forward+=(t1-t0)
            t_backward+=(t2-t1)
    
            t_forwards.append(t_forward)
            t_backwards.append(t_backward)
            
        # print(f"{epoch}: forward: {t_forward:.2f}s, backward: {t_backward:.2f}s")
    
    df_new_row = pd.DataFrame({
        'device': [device], 
        'samples_count': [samples_count], 
        'dataset_size': [samples_count * i_dim], 
        'params_count': [params_count], 
        'duration': [time.time()-t_start], 
        'avg. forward': [np.array(t_forwards).mean()], 
        'avg. backward': [np.array(t_backwards).mean()]
    })
    df2 = pd.concat([df2, df_new_row], ignore_index=True)
    
df2

  0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

,device,samples_count,dataset_size,params_count,duration,avg. forward,avg. backward
0,cpu,100,14400,57944,2.276265,0.023280,0.025933
1,cpu,10000,1440000,57944,21.742128,0.164033,0.373061
2,cpu,100000,14400000,57944,265.908312,2.814569,3.845342
3,cuda,100,14400,57944,4.853988,0.037764,0.066642
4,cuda,10000,1440000,57944,5.306885,0.041206,0.072706
5,cuda,100000,14400000,57944,7.492752,0.076247,0.102712
6,cuda,1000000,144000000,57944,62.370942,0.614556,0.946803


# Conv Model

In [10]:
df3 = pd.DataFrame()

In [11]:
class_names = ['airplane','automobile','bird','cat','deer', 'dog','frog','horse','ship','truck']
data_path = '/tmp/conv-model-test'
cifar10 = datasets.CIFAR10(
    data_path, train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4915, 0.4823, 0.4468),
                             (0.2470, 0.2435, 0.2616))
    ]))
cifar10 = [(img, label) for img, label in tqdm(cifar10)]
dataitem_size = next(iter(cifar10))[0].ravel().shape[0]
len(cifar10)

  0%|          | 0/50000 [00:00<?, ?it/s]

50000

In [12]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 8, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(8 * 8 * 8, 32)
        self.fc2 = nn.Linear(32, 10)
        
    def forward(self, x):
        out = F.max_pool2d(torch.tanh(self.conv1(x)), 2)
        out = F.max_pool2d(torch.tanh(self.conv2(out)), 2)
        out = out.view(-1, 8 * 8 * 8)
        out = torch.tanh(self.fc1(out))
        out = self.fc2(out)
        return out

In [13]:
Config = namedtuple('Config', 'device, samples_count, batch_size, epochs_count')
configs = [
    Config('cpu',   samples_count=50_000, batch_size=100,     epochs_count=5),
    Config('cpu',   samples_count=50_000, batch_size=1_000,   epochs_count=5),
    Config('cuda',  samples_count=50_000, batch_size=100,     epochs_count=5),
    Config('cuda',  samples_count=50_000, batch_size=1_000,   epochs_count=5),
]

In [15]:
for device, samples_count, batch_size, epochs_count in tqdm(configs):
    train_loader = torch.utils.data.DataLoader(cifar10[:samples_count], batch_size=batch_size, shuffle=True)
    net = Net().to(device=device)  
    params_count = sum([p.numel() for p in net.parameters()])
    optimizer = optim.SGD(net.parameters(), lr=1e-2)
    loss_fn = nn.CrossEntropyLoss()

    t_start = time.time()
    t_forwards = []
    t_backwards = []
    t_optisteps = []
    
    for epoch in tqdm(range(epochs_count), leave=False):
        t_forward, t_backward, t_optistep = 0, 0, 0
        
        for imgs, labels in tqdm(train_loader, leave=False):
            optimizer.zero_grad(set_to_none=True) # https://docs.pytorch.org/tutorials/recipes/recipes/tuning_guide.html
            
            imgs = imgs.to(device=device) 
            labels = labels.to(device=device)
            
            t0 = time.time()
            
            outputs = net(imgs)
            loss = loss_fn(outputs, labels)
            
            torch.cuda.synchronize()
            t1 = time.time()
            
            loss.backward()

            torch.cuda.synchronize()
            t2 = time.time()
            
            optimizer.step()

            torch.cuda.synchronize()
            t3 = time.time()
            
            t_forward += (t1-t0)
            t_backward += (t2-t1)
            t_optistep += (t3-t2)
    
            t_forwards.append(t_forward)
            t_backwards.append(t_backward)
            t_optisteps.append(t_optistep)

    df_new_row = pd.DataFrame({
        'device': [device], 
        'samples_count': [samples_count], 
        'batch_size': [batch_size], 
        'dataset_size': [samples_count * dataitem_size], 
        'params_count': [params_count], 
        'epochs_count': [epochs_count], 
        'duration': [time.time() - t_start], 
        'avg. forward': [np.array(t_forwards).mean()], 
        'avg. backward': [np.array(t_backwards).mean()],
        'avg. optistep': [np.array(t_optisteps).mean()]
    })
    df3 = pd.concat([df3, df_new_row], ignore_index=True)
    
df3

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

,device,samples_count,batch_size,dataset_size,params_count,epochs_count,duration,avg. forward,avg. backward,avg. optistep
0,cpu,50000,100,153600000,18354,5,22.286237,0.697133,1.260679,0.066584
1,cpu,50000,1000,153600000,18354,5,27.834729,1.071383,1.616771,0.008823
2,cuda,50000,100,153600000,18354,5,17.396020,0.432755,0.674313,0.125555
3,cuda,50000,1000,153600000,18354,5,2.719315,0.024719,0.078199,0.006019


In [ ]:
# train_loader = torch.utils.data.DataLoader(cifar2, batch_size=64, shuffle=True)

# for epoch in tqdm(range(1, n_epochs + 1)):
#   for imgs, labels in train_loader:
#     imgs = imgs.to(device=device) 
#     labels = labels.to(device=device)
#     imgs = imgs.to(device='cpu') 
#     labels = labels.to(device='cpu')
#     pass